<a href="https://colab.research.google.com/github/anupbagale/Whole-Brain-Segmentation/blob/main/notebooks/02_train_and_evaluate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2. Training and evaluation

Two 3D U-Nets are trained on the same subjects: one for all 45 classes and one for the 28-class reduced set.
Both are then tested at the original 1 mm resolution on subjects they never saw.

The question is whether asking the network for the 17 extra small structures costs accuracy on the
27 structures both models share.

Needs a GPU (Runtime > Change runtime type > T4). Training saves a checkpoint to Drive after every epoch,
so if Colab disconnects, run all cells again and it continues where it stopped.

In [ ]:
import os, sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("wbseg"):
        !git clone -q https://github.com/anupbagale/Whole-Brain-Segmentation.git
        %cd Whole-Brain-Segmentation
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
# Change these to where the data is on your Drive
IMAGE_DIR = "/content/drive/MyDrive/OASIS3_Dataset_Part4/Image_Folder"   # T1 images (.nii.gz)
LABEL_DIR = "/content/drive/MyDrive/OASIS3_Dataset_Part4/Label_Folder"   # FreeSurfer aseg label maps
LUT_PATH = "/content/drive/MyDrive/FreeSurferColorLUT.txt"               # optional, only for colours
RUN_DIR = "/content/drive/MyDrive/whole_brain_seg_runs"                   # checkpoints and predictions

if not os.path.exists(LUT_PATH):
    LUT_PATH = None
os.makedirs("figures", exist_ok=True)
os.makedirs("results", exist_ok=True)

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon
from monai.data import CacheDataset, DataLoader
from monai.utils import set_determinism

from wbseg.data import find_pairs, split_subjects, save_split, get_transforms
from wbseg.labels import LABEL_SETS, REDUCED_LABELS, colour_table
from wbseg.train import build_model, train
from wbseg.evaluate import evaluate, summarise, predict
from wbseg import plots

SEED = 0
SIZE = 128          # the whole head is resized to SIZE^3 for the network
N_VAL, N_TEST = 18, 30
MAX_EPOCHS = 150
set_determinism(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Split

Subjects are split once at random, and the same split is used for both models.

In [ ]:
pairs = find_pairs(IMAGE_DIR, LABEL_DIR)
split = split_subjects(pairs, n_val=N_VAL, n_test=N_TEST, seed=SEED)
save_split(split, "results/data_split.json")
print({k: len(v) for k, v in split.items()})

## Training

In [ ]:
histories = {}
for name, labels in LABEL_SETS.items():
    print(f"\n=== {name} label set: {len(labels)} classes ===")
    train_ds = CacheDataset(split["train"], get_transforms(labels, SIZE, train=True), num_workers=2)
    val_ds = CacheDataset(split["val"], get_transforms(labels, SIZE), num_workers=2)
    train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=1, num_workers=1)
    model = build_model(len(labels))
    histories[name] = train(model, train_loader, val_loader, len(labels),
                            out_dir=f"{RUN_DIR}/{name}", max_epochs=MAX_EPOCHS,
                            val_every=2, patience=15, device=device)
    del train_ds, val_ds

In [ ]:
fig = plots.training_curves({"45 classes": histories["full"], "28 classes": histories["reduced"]})
fig.savefig("figures/training_curves.png", dpi=150)
plt.show()

## Test set

Each test scan is segmented, the class scores are resized back to 256^3 and Dice is computed against the
original FreeSurfer label map. Predictions are saved to Drive with FreeSurfer label values, so they open in
any viewer (FreeView, ITK-SNAP) on top of the T1.

In [ ]:
results = {}
for name, labels in LABEL_SETS.items():
    model = build_model(len(labels)).to(device)
    model.load_state_dict(torch.load(f"{RUN_DIR}/{name}/best_model.pt", map_location=device))
    results[name] = evaluate(model, split["test"], labels, SIZE,
                             save_dir=f"{RUN_DIR}/{name}/test_predictions", device=device)
    results[name].to_csv(f"results/test_dice_{name}.csv", index=False)

In [ ]:
full_table = summarise(results["full"])
red_table = summarise(results["reduced"])
full_table.to_csv("results/dice_per_structure_full.csv", index=False)
red_table.to_csv("results/dice_per_structure_reduced.csv", index=False)
full_table

## Full vs reduced on the 27 shared structures

In [ ]:
shared = REDUCED_LABELS[1:]

def subject_means(res):
    r = res[res["label"].isin(shared)]
    return r.groupby("id")["dice"].mean()

a, b = subject_means(results["full"]), subject_means(results["reduced"])
a, b = a.loc[b.index], b
stat, p = wilcoxon(a, b)
comparison = pd.DataFrame({"45 classes": a, "28 classes": b})
print(comparison.describe().loc[["mean", "std", "min", "max"]].round(4))
print(f"\npaired difference (45 - 28): {(a - b).mean():+.4f}")
print(f"Wilcoxon signed-rank test over {len(a)} test subjects: p = {p:.3g}")

In [ ]:
fig = plots.dice_by_structure([full_table[full_table["label"].isin(shared)], red_table])
fig.savefig("figures/dice_shared_structures.png", dpi=150)
plt.show()

fig = plots.dice_by_structure([full_table], names=("45 classes",))
fig.savefig("figures/dice_all_structures.png", dpi=150)
plt.show()

## Example test subject

In [ ]:
import nibabel as nib
labels = LABEL_SETS["full"]
case = split["test"][0]
img = nib.as_closest_canonical(nib.load(case["image"]))
truth = nib.as_closest_canonical(nib.load(case["label"]))
pred = nib.as_closest_canonical(nib.load(f"{RUN_DIR}/full/test_predictions/{case['id']}_pred.nii.gz"))
fig = plots.comparison_figure(img.get_fdata(), np.asarray(truth.dataobj).astype(int),
                              np.asarray(pred.dataobj).astype(int), labels, colour_table(labels, LUT_PATH),
                              index=(100, 128, 140))
fig.savefig("figures/example_segmentation.png", dpi=150)
plt.show()

## Summary

Writes the main numbers to `results/summary.md`.

In [ ]:
def mean_sd(res, labs=None):
    r = res if labs is None else res[res["label"].isin(labs)]
    m = r.groupby("id")["dice"].mean()
    return f"{m.mean():.3f} ± {m.std():.3f}"

lines = [
    "# Test results", "",
    f"Split: {len(split['train'])} train, {len(split['val'])} validation, {len(split['test'])} test scans (seed {SEED}).", "",
    "Mean Dice per subject, averaged over structures, then mean ± SD over test subjects.", "",
    "| Model | All its structures | 27 shared structures |",
    "|---|---|---|",
    f"| 45 classes | {mean_sd(results['full'])} | {mean_sd(results['full'], shared)} |",
    f"| 28 classes | {mean_sd(results['reduced'])} | {mean_sd(results['reduced'], shared)} |", "",
    f"Paired difference on the shared structures: {(a - b).mean():+.4f} (Wilcoxon signed-rank p = {p:.3g}).", "",
    "Best five structures (45 classes): " + ", ".join(f"{s} {m:.3f}" for s, m in full_table[["structure", "mean"]].head(5).values),
    "", "Worst five structures (45 classes): " + ", ".join(f"{s} {m:.3f}" for s, m in full_table[["structure", "mean"]].tail(5).values),
]
open("results/summary.md", "w").write("\n".join(lines) + "\n")
print("\n".join(lines))